# Business Questions

**Dataset**: UK e-commerce gift shop, transactional records 2010–2011

**Business Questions**:
1. Which products are most popular by quantity sold?
2. Which countries generate the most revenue?

**Useful for**: Shop owner and marketing team to improve sales strategy.

# 01 Load_Cleaning_Preprocessing

Goal: inspect raw records and spot possible data issues.

Loading

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
e_co_raw_df = pd.read_csv('../data/e-commerce.csv', encoding='latin-1')


In [11]:
e_co_raw_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Obeservation 1: StockCode 84029E and 84029G share the same first 5 character the only differences is the last letter. However, they the record of the StockCodes have different descriptions.

Implication 1: However, we should prioritize other cleaning issues first as this is not relvant to our business question.

In [12]:
e_co_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


Observation 1: CustomerID has missing values.

Action 1: Explore missing customer patterns before cleaning.

Observation 2: Description has missing values.

Action 1: Explore missing product patterns before cleaning.

Observation 3: InvoiceDate is not datetime.

Action 1: Convert before time analysis.

Cleaning and preprocessing

Goal: Inspect missing patterns to reduce influence of missing records

In [13]:
#Changing the datatype of InvoiceDate from string to usable datetime format
e_co_raw_df['InvoiceDate']=pd.to_datetime(e_co_raw_df['InvoiceDate'])

In [ ]:
# Calculating and comparing the missing ratio of CustomerID and Description respectively
Missing_ratio_CustomerID=(e_co_raw_df['CustomerID'].isna().sum())/len(e_co_raw_df['CustomerID'])
Missing_ratio_Description=(e_co_raw_df['Description'].isna().sum())/len(e_co_raw_df['Description'])
print("The missing report:")
print(f"The number of missing records of CustomerID: {e_co_raw_df['CustomerID'].isna().sum()}")
print(f"The number of missing records of Description: {e_co_raw_df['Description'].isna().sum()}")
print()

print(f'The percentage of missing CustomerID: {Missing_ratio_CustomerID*100}')
print(f'The percentage of missing Description: {Missing_ratio_Description*100}')

The missing report:
The number of missing records of CustomerID: 135080
The number of missing records of CustomerID: 1454

The percentage of missing CustomerID: 24.926694334288598
The percentage of missing Description: 0.2683107311375157


Observation 1: CustomerID has about 25% missing values.

Action 1: Filter missing IDs first, then inspect their patterns.

Action 2: Compare later only if the missing group shows a clear clue.

Observation 2: Description has only a few missing values.

Action 1: Filter missing Descriptions first, then inspect their patterns.

Action 2: Compare later only if the missing group shows a clear clue.

In [15]:
#Filtering out all the 'CustomerID' missing records in.
CustomerID_NaN=e_co_raw_df[e_co_raw_df['CustomerID'].isna()]
CustomerID_NaN.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom


Obeservation 1: The first record as both description and customerID NaN and the unitprice is 0.0.

Meaning 1: This might indicates a record of gift, a failed transaction 

Meaning 2: The loss of description might paired to loss of CustomerID.

Action 1: Look for the patterns of missing descriptions to identify the paired pattern in meaning 2



Observation 2: The 5 records are on the same day

Meaning: This might indicate a server down on December 1st or even December 2010 due to the crowd on Christmas Eve or a guest check-in day.

Action 1: Make a line chart of dates to explore whether December has the highest amount of missing records.


Count missing CustomerID by date.

would be very long the 
and hard to identify patterns 

so  I use a time series line chart

Or else 
I will do the following
1. turn all the datetime into months
2. value count those months and only show top 5
3. if december is the largest,  filter out december 
and look for the 31 days and value count the top 5t to disccover whether the first date of 

In [ ]:
# To view whether how long does the record spans
e_co_raw_df['InvoiceDate'].agg(['min', 'max'])

min   2010-12-01 08:26:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[us]

In [21]:
e_co_raw_df['month']=e_co_raw_df['InvoiceDate'].dt.month

In [22]:
e_co_raw_df['month'].value_counts().sort_values(ascending=False).head()

month
11    84711
12    68006
10    60742
9     50226
7     39518
Name: count, dtype: int64

In [ ]:
#The filtering out the missing description
Description_NaN=e_co_raw_df[e_co_raw_df['Description'].isna()]

In [ ]:
#Investigating whether description NaN records are the subsets of CustomerID NaNs
Description_NaN.isna().sum()

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID     1454
Country           0
dtype: int64

Obeservation 1: When Descriptions are NaN customerID are NaN so Descriptions NaNs are the subset of CustomerID.

Action 1: This might be due to that the customer has rejected the transanction or the product has been quickly put down from the shelf due to ?

# 02 Transformation

# 03 Analyse

# 04 Insight and Conclusion